In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/processed/cleaned_churn.csv")

df.head()

In [ ]:
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Churn rate:", df["Churn"].mean())

In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

categorical_cols= df.select_dtypes(include=['string','object','category']).columns.tolist()

print(numeric_cols)
print(categorical_cols)

In [ ]:
df["TenureGroup"]= pd.cut(
    df["tenure"],
    bins=[-1,6,12,24,48,72],
    labels=[
        '0-6 Months',
        '7-12 Months',
        '13-24 Months',
        '25-48 Months',
        '49-72 Months'
    ]
)

sns.barplot(
    data=df,
    x="TenureGroup",
    y="Churn",
    hue="TenureGroup",
    palette="Set2",
    legend=False,
    errorbar=None
)

plt.xticks(rotation=45)
plt.title("Churn Rate by Tenure Group")
plt.show()

In [ ]:
service_cols = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["TotalServices"] = (df[service_cols] == "Yes").sum(axis=1)

In [ ]:
sns.barplot(
    data=df,
    x="TotalServices",
    y="Churn",
    errorbar=None
)

plt.title("Churn Rate by Total Services")
plt.show()

In [ ]:
df["AverageMonthlySpend"] = (
    df["TotalCharges"] / df["tenure"].replace(0, np.nan)
)

df["AverageMonthlySpend"] = df["AverageMonthlySpend"].fillna(
    df["MonthlyCharges"]
)

In [ ]:
df["IsNewCustomer"] = (df["tenure"] <= 6).astype(int)

In [ ]:
df["HasSupport"] = (
    (df["TechSupport"] == "Yes") |
    (df["OnlineSecurity"] == "Yes")
).astype(int)

In [ ]:
df["HasFamily"] = (
    (df["Partner"] == "Yes") |
    (df["Dependents"] == "Yes")
).astype(int)

In [ ]:
df["ChargesPerService"] = (
    df["MonthlyCharges"] /
    df["TotalServices"].clip(lower=1)
)

In [ ]:
new_features = [
    "TenureGroup",
    "TotalServices",
    "AverageMonthlySpend",
    "IsNewCustomer",
    "HasSupport",
    "HasFamily",
    "ChargesPerService"
]

df[new_features].head(10)

In [ ]:
print(df.isnull().sum())

print(
    np.isinf(
        df.select_dtypes(include="number")
    ).sum()
)

print(df.shape)

In [ ]:
df.to_csv(
    "../data/processed/engineered_churn.csv",
    index=False
)